# Fitting a line: closed form vs gradient descent
We fit $\hat{y} = wx + b$ by minimizing the mean squared error, in two ways, and compare them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
TRUE_W, TRUE_B = 2.5, 4.0
x = rng.uniform(0, 10, 100)
y = TRUE_W * x + TRUE_B + rng.normal(0, 2, 100)

plt.scatter(x, y, s=15)
plt.xlabel("x"); plt.ylabel("y"); plt.title("Our data")
plt.show()

In [ ]:
def mse(w, b, x, y):
    return np.mean((w * x + b - y) ** 2)

## 1. Closed-form solution
$$w = \frac{\sum (x_i-\bar{x})(y_i-\bar{y})}{\sum (x_i-\bar{x})^2}, \qquad b = \bar{y} - w\bar{x}$$

In [ ]:
w_cf = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean()) ** 2)
b_cf = y.mean() - w_cf * x.mean()
print(f"w = {w_cf:.4f}, b = {b_cf:.4f}, MSE = {mse(w_cf, b_cf, x, y):.4f}")

# The matrix form gives the same answer
X = np.column_stack([x, np.ones_like(x)])
theta = np.linalg.lstsq(X, y, rcond=None)[0]
print("lstsq:", theta)
assert np.allclose(theta, [w_cf, b_cf])

## 2. Gradient descent
$$\frac{\partial J}{\partial w} = \frac{2}{n}\sum r_i x_i, \qquad \frac{\partial J}{\partial b} = \frac{2}{n}\sum r_i, \qquad r_i = wx_i + b - y_i$$

In [ ]:
def gradient_descent(x, y, lr, steps):
    w, b = 0.0, 0.0
    history = [(w, b, mse(w, b, x, y))]
    for _ in range(steps):
        r = w * x + b - y
        grad_w = 2 * np.mean(r * x)
        grad_b = 2 * np.mean(r)
        w -= lr * grad_w
        b -= lr * grad_b
        history.append((w, b, mse(w, b, x, y)))
    return w, b, np.array(history)

w_gd, b_gd, hist = gradient_descent(x, y, lr=0.02, steps=2000)
print(f"w = {w_gd:.4f}, b = {b_gd:.4f}, MSE = {mse(w_gd, b_gd, x, y):.4f}")

## 3. Compare

In [ ]:
print(f"{'method':<20}{'w':>10}{'b':>10}{'MSE':>10}")
for name, w, b in [("true values", TRUE_W, TRUE_B),
                   ("closed form", w_cf, b_cf),
                   ("gradient descent", w_gd, b_gd)]:
    print(f"{name:<20}{w:>10.4f}{b:>10.4f}{mse(w, b, x, y):>10.4f}")

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
grid = np.linspace(0, 10, 100)
left.scatter(x, y, s=15, color="gray")
left.plot(grid, w_cf * grid + b_cf, lw=4, label="closed form")
left.plot(grid, w_gd * grid + b_gd, "--", lw=2, label="gradient descent")
left.legend(); left.set_title("Fitted lines")
right.plot(hist[:, 2], label="gradient descent")
right.axhline(mse(w_cf, b_cf, x, y), color="red", ls="--", label="closed-form minimum")
right.set_yscale("log"); right.set_xlabel("step"); right.set_ylabel("MSE")
right.legend(); right.set_title("Loss over time")
plt.tight_layout()
plt.show()

The true values differ slightly from both fits because the data is noisy.

## 4. The learning rate
Gradient descent is stable here only if $\eta < 2/\lambda_{\max}$, where $\lambda_{\max}$ is the largest eigenvalue of the Hessian $\frac{2}{n}X^\top X$.

In [ ]:
H = 2 / len(x) * X.T @ X
limit = 2 / np.linalg.eigvalsh(H).max()
print(f"largest stable learning rate: {limit:.4f}")

for factor in [0.05, 0.7, 1.05]:
    lr = factor * limit
    _, _, h = gradient_descent(x, y, lr, steps=100)
    plt.plot(h[:, 2], label=f"lr = {lr:.4f}")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("MSE"); plt.legend()
plt.show()

## 5. Feature scaling
Standardize `x`, run gradient descent again, then convert the parameters back.

In [ ]:
mu, s = x.mean(), x.std()
z = (x - mu) / s
w_z, b_z, hist_z = gradient_descent(z, y, lr=0.1, steps=100)
w_back, b_back = w_z / s, b_z - w_z * mu / s
print(f"after only 100 steps: w = {w_back:.4f}, b = {b_back:.4f}")

def steps_needed(history, tol=1e-6):
    best = mse(w_cf, b_cf, x, y)
    close = np.where(history[:, 2] - best < tol)[0]
    return close[0] if len(close) else None

print("steps to reach the minimum, raw x:         ", steps_needed(hist))
print("steps to reach the minimum, standardized x:", steps_needed(hist_z))

## Exercises
1. Change the noise level. How do the fitted values change?
2. Add one large outlier to `y`. What happens to the line, and why?
3. Implement stochastic gradient descent using one random point per step. Plot its loss next to full gradient descent.
4. Fit a quadratic by adding an `x**2` column to `X`.

In [ ]:
# Your code here